# AI Agent Autopsy - 04 LangGraph

This shows a simple example of LangGraph

The first example below comes from Chapter 2 of the book it just shows how to create a Chatbot with the LangGraph functions.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model

In [ ]:
class State(TypedDict):
   messages: Annotated[list, add_messages]

graph = StateGraph(State)

In [ ]:
model = init_chat_model("google_genai:gemini-2.0-flash")

def chatbot(state: State):
   return {"messages": [model.invoke(state["messages"])]}

graph.add_node("chatbot", chatbot)

In [ ]:
from langgraph.graph import START

graph.add_edge(START, "chatbot")

app = graph.compile()


In [ ]:
question = “What is a language model?”
for event in graph.stream(
      {"messages": [{"role": "user", "content": question}]}
   ):
      for value in event.values():
         print("Assistant:", value["messages"][-1].content)


The code above produced the result shown in the book of a simple description of a language model.

## Reflexion Pattern

To really see why LangGraph exists as a library for building agents we need to introduce the capacity for a loop in the agent reasoning path. Allowing for this computational flow is the reason we need to use a Graph in the system design.

The simplest looping agentic patten is a reflexion pattern, in which one agent generates a response and a second evaluates it.

In [26]:
import os
import re
from typing import TypedDict, Literal

from openai import OpenAI
from langgraph.graph import StateGraph, END

In [27]:
MODEL = "gpt-4o"
MAX_ITERATIONS = 5

client = OpenAI()

In [28]:
class State(TypedDict):
    question: str
    draft: str
    feedback: str
    verdict: Literal["approve", "revise", ""]
    iterations: int

In [29]:
def responder(state: State) -> dict:
    """Generate (or revise) an answer to the user's question."""
    #...
    return {
        "draft": draft,
        "iterations": state["iterations"] + 1,
    }

In [30]:
def reviewer(state: State) -> dict:
    """Evaluate the draft. Emit VERDICT: APPROVE or REVISE plus feedback."""
    #...
    return {"verdict": verdict, "feedback": feedback}

In [31]:
def route(state: State) -> Literal["responder", "__end__"]:
    if state["verdict"] == "approve":
        return END
    if state["iterations"] >= MAX_ITERATIONS:
        print(f"\n[stopping: hit max iterations = {MAX_ITERATIONS}]")
        return END
    return "responder"

def build_graph():
    g = StateGraph(State)
    g.add_node("responder", responder)
    g.add_node("reviewer", reviewer)
    g.set_entry_point("responder")
    g.add_edge("responder", "reviewer")
    g.add_conditional_edges("reviewer", route, {"responder": "responder", END: END})
    return g.compile()

In [32]:
def answer_question(question: str) -> str:
    app = build_graph()
    initial: State = {
        "question": question,
        "draft": "",
        "feedback": "",
        "verdict": "",
        "iterations": 0,
    }
    final = app.invoke(initial)
    return final["draft"]

In [33]:
question = (
        "What were the main technical reasons Concorde was retired in 2003, "
        "and what specific factors would prevent a direct successor today?"
)
print(f"Question: {question}")
final_answer = answer_question(question)
print("\n=== Final approved answer ===")
print(final_answer)

Question: What were the main technical reasons Concorde was retired in 2003, and what specific factors would prevent a direct successor today?

--- Responder (iteration 1) ---
Concorde was retired in 2003 for several technical reasons, compounded by economic and regulatory factors. Here are the main technical reasons for its retirement:

1. **Aging and Maintenance Challenges**: By 2003, the Concorde fleet was over 25 years old. Aging aircraft required increasingly extensive and costly maintenance to ensure safety and reliability. Many of its components, such as avionics and engines, were outdated, and sourcing replacement parts became challenging, as the original manufacturers had ceased production or were no longer available.

2. **Fuel Inefficiency**: Concorde's turbojet engines, primarily the Rolls-Royce/Snecma Olympus 593, were designed for supersonic speeds, which inherently led to high fuel consumption compared to subsonic aircraft. The aircraft's economics were substantially aff

## Use the LangGraph State engine for purely deterministic process

This is an interesting example I pulled from a blog post that shows how the Graph computation engine of LangGraph can be used for purely deterministic processes, and can be easily visualised. Thanks to [Ander Fernández Jauregui](https://anderfernandez.com/) for this one. You can see his blog post here: [Creating an Agent-Based System with LangGraph](https://anderfernandez.com/en/blog/agent-systems-with-langgraph/)

In [5]:
from pydantic import BaseModel

class GraphState(BaseModel):
    graph_state: str
    number_interactions: int = 0
    verbose: bool = True

In [6]:
from langgraph.graph import StateGraph

graph = StateGraph(GraphState)

In [7]:
def greeting_node(state):
    if state.verbose:
        print("---- Greeting Node ----")
    state.graph_state += "Hello! "
    state.number_interactions += 1
    return state


def normal_node(state):
    if state.verbose:
        print("---- Normal Node ----")
    state.graph_state += "Today you should: "
    state.number_interactions += 1
    return state


def plan_node1(state):
    if state.verbose:
        print("---- Plan Node 1 ----")
    state.graph_state += "go bowling!"
    state.number_interactions += 1
    return state


def plan_node2(state):
    if state.verbose:
        print("---- Plan Node 2 ----")
    state.graph_state += "Netflix & chill"
    state.number_interactions += 1
    return state


graph.add_node("greeting_node", greeting_node)
graph.add_node("normal_node", normal_node)
graph.add_node("plan_node1", plan_node1)
graph.add_node("plan_node2", plan_node2)

In [8]:
import random

from typing import Literal
from langgraph.graph import START, END


def decide_plan(state) -> Literal["plan_node1", "plan_node2"]:
    return 'plan_node1' if random.random() < 0.5 else 'plan_node2'


graph.add_edge(START, "greeting_node")
graph.add_edge("greeting_node", "normal_node")

graph.add_conditional_edges("normal_node", decide_plan)
graph.add_edge("plan_node1", END)
graph.add_edge("plan_node2", END)

In [9]:
graph_compiled = graph.compile()

In [11]:
_ = (
    graph_compiled
    .get_graph()
    .draw_mermaid_png(output_file_path='images/simple_graph.png')
)